In [1]:
import os

os.makedirs("../csv", exist_ok=True)
os.makedirs("../svg", exist_ok=True)

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scienceplots

In [3]:
plt.rcParams.update({
    'font.family': 'TeX Gyre Termes',
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
})

In [4]:
TEXTWIDTH_PT = 426.79137
TEXTHEIGHT_PT = 702.78308
PT_PER_INCH = 72.27

FIG_WIDTH = TEXTWIDTH_PT / PT_PER_INCH
FIG_HEIGHT_1 = TEXTHEIGHT_PT / PT_PER_INCH
FIG_HEIGHT_2 = FIG_HEIGHT_1 / 2
FIG_HEIGHT_3 = FIG_HEIGHT_1 / 3
FIG_HEIGHT_4 = FIG_HEIGHT_1 / 4
FIG_HEIGHT_5 = FIG_HEIGHT_1 / 5
FIG_HEIGHT_6 = FIG_HEIGHT_1 / 6
FIG_HEIGHT_7 = FIG_HEIGHT_1 / 7
FIG_HEIGHT_8 = FIG_HEIGHT_1 / 8

In [5]:
backendy_pl = {
    'scalar': 'skalarny',
    'sse2': 'SSE2',
    'avx2': 'AVX2',
    'avx512': 'AVX-512',
    'neon': 'NEON',
}

In [6]:
wersje_pl = {
    'Speck32_64':   '32/64',
    'Speck48_72':   '48/72',
    'Speck48_96':   '48/96',
    'Speck64_96':   '64/96',
    'Speck64_128':  '64/128',
    'Speck96_96':   '96/96',
    'Speck96_144':  '96/144',
    'Speck128_128': '128/128',
    'Speck128_192': '128/192',
    'Speck128_256': '128/256',
}

In [7]:
backend_order = ['scalar', 'sse2', 'avx2', 'avx512']
version_order = ['32_64', '48_72', '48_96', '64_96', '64_128', '96_96', '96_144', '128_128', '128_192', '128_256']

In [8]:
system_x86 = pd.read_csv('../data/system_x86.csv')
system_aarch64 = pd.read_csv('../data/system_aarch64.csv')

In [9]:
system_x86['time_per_key_ns'] = system_x86['duration_ns'] / system_x86['throughput_num']
system_x86['keys_per_sec'] = 1e9 / system_x86['time_per_key_ns']

print(system_x86)

      bits_measured benchmark backend architecture cipher_mode  \
0                12    system  Avx512       x86_64         Ecb   
1                13    system  Avx512       x86_64         Ecb   
2                14    system  Avx512       x86_64         Ecb   
3                15    system  Avx512       x86_64         Ecb   
4                16    system  Avx512       x86_64         Ecb   
...             ...       ...     ...          ...         ...   
1115             22    system  Scalar       x86_64         Cbc   
1116             23    system  Scalar       x86_64         Cbc   
1117             24    system  Scalar       x86_64         Cbc   
1118             28    system  Scalar       x86_64         Cbc   
1119             32    system  Scalar       x86_64         Cbc   

             function       version  suffix  throughput_num unit  duration_ns  \
0     EncryptInflight    Speck32_64       1            4096   ns       752077   
1     EncryptInflight    Speck32_64       1  

In [10]:
system_aarch64

,bits_measured,benchmark,backend,architecture,cipher_mode,function,version,suffix,throughput_num,unit,duration_ns
0,11,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,2048,ns,751792
1,12,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,4096,ns,330041
2,13,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,8192,ns,302209
3,14,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,16384,ns,260625
4,15,system,Neon,aarch64,Ecb,EncryptInflight,Speck32_64,1,32768,ns,328292
...,...,...,...,...,...,...,...,...,...,...,...
555,21,system,Scalar,aarch64,Cbc,EncryptInflight,Speck128_256,2,2097152,ns,10909334
556,22,system,Scalar,aarch64,Cbc,EncryptInflight,Speck128_256,2,4194304,ns,20205125
557,23,system,Scalar,aarch64,Cbc,EncryptInflight,Speck128_256,2,8388608,ns,39700791
558,27,system,Scalar,aarch64,Cbc,EncryptInflight,Speck128_256,2,134217728,ns,614227625


In [11]:
import pandas as pd
import numpy as np
from scipy import stats

system_x86 = pd.read_csv('../data/system_x86.csv')

GROUP_COLS = ['backend', 'function', 'version', 'suffix', 'cipher_mode']

def key_bits_from_version(v):
    return int(str(v).split('_')[1])

def fit_group(g):
    g = g.sort_values('throughput_num')
    n = g['throughput_num'].to_numpy(dtype=float)
    t = g['duration_ns'].to_numpy(dtype=float)

    if len(g) < 3:
        return None

    kb = key_bits_from_version(g['version'].iloc[0])
    N_full = 2.0 ** kb

    slope, intercept, lo, hi = stats.theilslopes(t, n)
    t_key = slope
    thr = 1e9 / t_key if t_key > 0 else np.nan

    tau, p_tau = stats.kendalltau(n, t)

    pred_total_ns = intercept + slope * N_full
    pred_total_s = pred_total_ns * 1e-9
    span = n.max() / n.min()

    return {
        'key_bits': kb,
        'overhead_ns': intercept,
        'ns_per_key': t_key,
        'ns_per_key_lo': 1e9 / hi if hi > 0 else np.nan,
        'ns_per_key_hi': 1e9 / lo if lo > 0 else np.nan,
        'throughput_keys_per_s': thr,
        'kendall_tau': tau,
        'p_value_kendall': p_tau,
        'full_keyspace': N_full,
        'pred_total_seconds': pred_total_s,
        'pred_total_years': pred_total_s / (3600 * 24 * 365.25),
        'reliable': (t_key > 0) and (span >= 8) and (p_tau < 0.05),
    }

rows = []
for keys, g in system_x86.groupby(GROUP_COLS):
    res = fit_group(g)
    if res is not None:
        rows.append(dict(zip(GROUP_COLS, keys)) | res)

predictions = pd.DataFrame(rows).sort_values(GROUP_COLS).reset_index(drop=True)
predictions

,backend,function,version,suffix,cipher_mode,key_bits,overhead_ns,ns_per_key,ns_per_key_lo,ns_per_key_hi,throughput_keys_per_s,kendall_tau,p_value_kendall,full_keyspace,pred_total_seconds,pred_total_years,reliable
0,Avx2,EncryptInflight,Speck128_128,1,Cbc,128,1.485147e+05,2.032418,4.142675e+08,2.441833e+09,4.920248e+08,0.714286,0.030159,3.402824e+38,6.915960e+29,2.191535e+22,True
1,Avx2,EncryptInflight,Speck128_128,1,Ecb,128,1.625177e+05,1.168160,4.102388e+08,9.080306e+08,8.560468e+08,1.000000,0.000397,3.402824e+38,3.975044e+29,1.259615e+22,True
2,Avx2,EncryptInflight,Speck128_128,2,Cbc,128,1.002810e+06,1.113687,8.709363e+08,9.163335e+08,8.979181e+08,1.000000,0.000397,3.402824e+38,3.789681e+29,1.200878e+22,True
3,Avx2,EncryptInflight,Speck128_128,2,Ecb,128,-1.738095e+04,1.121872,8.009989e+08,9.149155e+08,8.913670e+08,1.000000,0.000397,3.402824e+38,3.817534e+29,1.209703e+22,True
4,Avx2,EncryptInflight,Speck128_192,1,Cbc,192,1.578010e+05,1.095000,8.914167e+08,2.571249e+09,9.132416e+08,0.714286,0.030159,6.277102e+57,6.873430e+48,2.178058e+41,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,Sse2,EncryptInflight,Speck96_144,2,Ecb,144,1.466030e+05,2.689566,3.536931e+08,3.733305e+08,3.718072e+08,1.000000,0.000397,2.230075e+43,5.997932e+34,1.900630e+27,True
156,Sse2,EncryptInflight,Speck96_96,1,Cbc,96,1.662294e+05,2.768299,3.490918e+08,4.094362e+08,3.612327e+08,1.000000,0.000397,7.922816e+28,2.193272e+20,6.950061e+12,True
157,Sse2,EncryptInflight,Speck96_96,1,Ecb,96,1.708735e+05,2.747331,3.624955e+08,4.020258e+08,3.639897e+08,0.904762,0.002778,7.922816e+28,2.176660e+20,6.897418e+12,True
158,Sse2,EncryptInflight,Speck96_96,2,Cbc,96,-2.047894e+05,2.647349,3.703078e+08,3.838761e+08,3.777363e+08,1.000000,0.000397,7.922816e+28,2.097446e+20,6.646406e+12,True


In [12]:
import os
import numpy as np

INDEX_COLS = ["backend", "version", "cipher_mode"]
SUFFIXES   = sorted(predictions["suffix"].unique())

def fmt_pow2(x):
    if pd.isna(x) or x <= 0:
        return "--"
    return rf"\(2^{{{np.log2(x):.2f}}}\)"

def fmt_sci(x):
    if pd.isna(x):
        return "--"
    m, e = f"{x:.3e}".split("e")
    return rf"\({float(m):.2f}\times10^{{{int(e)}}}\)"

wide = predictions.set_index(INDEX_COLS + ["suffix"])

cols = {}
for suf in SUFFIXES:
    sub = wide.xs(suf, level="suffix")
    cols[f"{suf}_throughput"] = sub["throughput_keys_per_s"].map(fmt_pow2)
    cols[f"{suf}_years"]      = sub["pred_total_years"].map(fmt_sci)

latex_df = pd.DataFrame(cols).reset_index()

# --- klucze sortujące z SUROWYCH wartości (przed tłumaczeniem) ---
latex_df["_bk"] = pd.Categorical(
    latex_df["backend"].astype(str).str.lower(),
    categories=backend_order, ordered=True)
latex_df["_ver"] = pd.Categorical(
    latex_df["version"].astype(str).str.replace("Speck", "", regex=False),
    categories=version_order, ordered=True)

latex_df = latex_df.sort_values(["_bk", "_ver"]).drop(columns=["_bk", "_ver"])

# --- dopiero teraz tłumaczenia na tekst ---
latex_df["backend"] = latex_df["backend"].astype(str).str.lower().map(backendy_pl).fillna(latex_df["backend"])
latex_df["version"] = latex_df["version"].map(wersje_pl).fillna(latex_df["version"])

latex_df = latex_df.reset_index(drop=True)

os.makedirs("../csv", exist_ok=True)
latex_df.to_csv("../csv/prediction_results.csv", index=False)
latex_df

,backend,version,cipher_mode,1_throughput,1_years,2_throughput,2_years
0,skalarny,32/64,Cbc,\(2^{29.35}\),\(8.53\times10^{2}\),\(2^{29.36}\),\(8.47\times10^{2}\)
1,skalarny,32/64,Ecb,\(2^{29.37}\),\(8.43\times10^{2}\),\(2^{29.36}\),\(8.47\times10^{2}\)
2,skalarny,48/72,Cbc,\(2^{28.36}\),\(4.33\times10^{5}\),\(2^{28.30}\),\(4.53\times10^{5}\)
3,skalarny,48/72,Ecb,\(2^{28.37}\),\(4.33\times10^{5}\),\(2^{28.33}\),\(4.44\times10^{5}\)
4,skalarny,48/96,Cbc,\(2^{28.35}\),\(7.34\times10^{12}\),\(2^{28.19}\),\(8.18\times10^{12}\)
...,...,...,...,...,...,...,...
75,AVX-512,128/128,Ecb,\(2^{29.75}\),\(1.19\times10^{22}\),\(2^{30.13}\),\(9.18\times10^{21}\)
76,AVX-512,128/192,Cbc,\(2^{29.80}\),\(2.12\times10^{41}\),\(2^{30.17}\),\(1.65\times10^{41}\)
77,AVX-512,128/192,Ecb,\(2^{29.52}\),\(2.58\times10^{41}\),\(2^{30.14}\),\(1.68\times10^{41}\)
78,AVX-512,128/256,Cbc,\(2^{29.65}\),\(4.35\times10^{60}\),\(2^{30.07}\),\(3.25\times10^{60}\)


In [13]:
import numpy as np

# różnica log2 throughputu: dodatnia => suffix 2 szybszy
piv = predictions.pivot_table(
    index=["backend", "version"],
    columns="suffix",
    values="throughput_keys_per_s",
    observed=True,
)

cmp = piv.reset_index()
cmp.columns = ["backend", "version", "thr_s1", "thr_s2"]

cmp["delta_log2"]        = np.log2(cmp["thr_s2"] / cmp["thr_s1"])   # +1.0 = 2x szybciej
cmp["speedup_s2_over_s1"] = cmp["thr_s2"] / cmp["thr_s1"]           # krotność
cmp["winner"] = np.where(cmp["delta_log2"] > 0, "suffix 2",
                np.where(cmp["delta_log2"] < 0, "suffix 1", "remis"))

suffix_analysis = cmp.sort_values("delta_log2", ascending=False).reset_index(drop=True)
suffix_analysis

,backend,version,thr_s1,thr_s2,delta_log2,speedup_s2_over_s1,winner
0,Avx512,Speck32_64,1.670578e+09,6.778813e+09,2.020685,4.057765,suffix 2
1,Avx2,Speck32_64,1.658659e+09,3.995330e+09,1.268297,2.408771,suffix 2
2,Avx512,Speck64_128,1.625985e+09,2.786067e+09,0.776916,1.713464,suffix 2
3,Avx512,Speck64_96,1.791318e+09,2.925437e+09,0.707631,1.633120,suffix 2
4,Avx512,Speck128_192,8.533271e+08,1.193333e+09,0.483826,1.398448,suffix 2
5,Avx2,Speck128_128,6.740358e+08,8.946426e+08,0.408486,1.327292,suffix 2
6,Sse2,Speck64_128,9.116126e+08,1.203641e+09,0.400912,1.320342,suffix 2
7,Avx2,Speck96_96,5.404806e+08,7.119297e+08,0.397492,1.317216,suffix 2
8,Avx2,Speck128_192,6.870866e+08,8.984794e+08,0.386993,1.307665,suffix 2
9,Avx2,Speck96_144,5.429571e+08,7.040042e+08,0.374746,1.296611,suffix 2


In [14]:
wins = suffix_analysis["winner"].value_counts()

summary = pd.DataFrame({
    "metryka": [
        "wierszy ogółem", "wygrane suffix 2", "wygrane suffix 1", "remisy",
        "śr. przewaga suffix 2 [×]", "mediana przewagi [×]",
        "maks. przewaga suffix 2 [×]", "maks. przewaga suffix 1 [×]",
    ],
    "wartość": [
        len(suffix_analysis),
        int(wins.get("suffix 2", 0)),
        int(wins.get("suffix 1", 0)),
        int(wins.get("remis", 0)),
        2.0 ** suffix_analysis["delta_log2"].mean(),
        2.0 ** suffix_analysis["delta_log2"].median(),
        suffix_analysis["speedup_s2_over_s1"].max(),
        1.0 / suffix_analysis["speedup_s2_over_s1"].min(),
    ],
})

per_backend = (suffix_analysis
    .groupby("backend", observed=True)["delta_log2"]
    .agg(["mean", "median", "min", "max"])
    .assign(speedup_mean=lambda d: 2.0 ** d["mean"]))

per_backend

,mean,median,min,max,speedup_mean
backend,,,,,
Avx2,0.280752,0.212276,-0.114125,1.268297,1.214828
Avx512,0.508506,0.259123,0.072764,2.020685,1.422576
Scalar,0.045865,-0.015793,-0.068523,0.327025,1.032302
Sse2,0.089623,0.110748,-0.257793,0.400912,1.064092
